# TrashCAN 1.0 Exploration (GENERALIZATION TESTING)
**Assigned to:** ___
**No GPU needed.**

Used AFTER training to test generalization on unseen data. Not for training.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DATASET = "/content/drive/MyDrive/underwater_datasets/TrashCAN 1.0"
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "trashcan" in n.lower() and "icra" not in n.lower():
            DATASET = os.path.join(base, n)
            break
assert os.path.exists(DATASET), f"Not found. Available: {os.listdir(base)}"
print(f"✓ {DATASET}")


In [ ]:
from pathlib import Path; from collections import Counter
root = Path(DATASET)
for dp, dn, fn in sorted(os.walk(root)):
    depth = str(dp).replace(str(root),"").count(os.sep)
    if depth > 3: continue
    indent = "  "*depth
    exts = Counter(Path(f).suffix.lower() for f in fn)
    print(f"{indent}{os.path.basename(dp)}/")
    for e,c in sorted(exts.items()): print(f"{indent}  {c:>6} {e}")


In [ ]:
# === DETECT ANNOTATION FORMAT ===
from pathlib import Path
root = Path(DATASET)
txts = list(root.rglob("*.txt")); xmls = list(root.rglob("*.xml"))
jsons = list(root.rglob("*.json")); csvs = list(root.rglob("*.csv"))
print(f".txt: {len(txts)} | .xml: {len(xmls)} | .json: {len(jsons)} | .csv: {len(csvs)}")

if jsons:
    import json
    with open(jsons[0]) as f: data = json.load(f)
    if isinstance(data, dict):
        print(f"\nJSON keys: {list(data.keys())[:10]}")
        if "categories" in data:
            print(f"Categories: {data['categories'][:15]}")
        if "annotations" in data:
            print(f"Annotations: {len(data['annotations'])}")

if txts:
    print(f"\nSample .txt ({txts[0].name}):")
    for i,l in enumerate(open(txts[0])):
        if i<5: print(f"  {l.strip()}")


In [ ]:
# === CLASS DISTRIBUTION ===
from collections import Counter
from pathlib import Path
root = Path(DATASET)
counts = Counter()

# Try YOLO
for t in root.rglob("*.txt"):
    try:
        for l in open(t):
            p = l.strip().split()
            if len(p)>=5: counts[int(p[0])] += 1
    except: pass

# Try COCO JSON
if not counts:
    import json
    for j in root.rglob("*.json"):
        try:
            d = json.load(open(j))
            if "annotations" in d:
                for a in d["annotations"]: counts[a.get("category_id",-1)] += 1
        except: pass

if counts:
    print(f"Classes found: {len(counts)} | Total boxes: {sum(counts.values())}")
    for c in sorted(counts): print(f"  Class {c}: {counts[c]}")
else:
    print("Could not parse annotations")


In [ ]:
# === SAMPLE IMAGES ===
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
root = Path(DATASET)
imgs = sorted(root.rglob("*.jpg")) + sorted(root.rglob("*.png"))
if not imgs: print("No images!")
else:
    np.random.seed(42)
    picks = [imgs[i] for i in np.random.choice(len(imgs), min(6,len(imgs)), replace=False)]
    fig, axes = plt.subplots(2,3, figsize=(18,12))
    for i,p in enumerate(picks):
        ax = axes[i//3][i%3]
        img = cv2.imread(str(p))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{p.parent.name}/{p.name}\n({img.shape[1]}x{img.shape[0]})", fontsize=8)
        ax.axis("off")
    plt.suptitle("TrashCAN 1.0 Samples", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.savefig("/content/trashcan_samples.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
print('''
================================================================
TRASHCAN 1.0 SUMMARY
================================================================
1. Total images: ___
2. Annotation format: YOLO / COCO / VOC / None
3. Classes: ___
4. Overlaps with Trash-ICRA19? ___
5. Can compute mAP? ___
6. Image sizes: ___
================================================================
''')
